# 05 — Market intelligence: regime characterization + lead-lag structure

Two descriptive-intelligence builds that deepen what the research actually found,
turning the dashboard from "charts + a flag" into decision-support.

**Part 1 — Characterize the volatility regime.** The vol-regime flag was the one
predictable signal (holdout AUC 0.93). A binary flag is thin; a *characterized*
regime is useful: which assets drive it, how long it lasts, how much lead time
it gives.

**Part 2 — Lead-lag structure.** The correlation heatmap is contemporaneous. This
asks *who moves first* — when X moves this month, which commodities tend to
follow next month. Descriptive and honest (not price prediction), but genuinely
actionable context.

Explore window for structure-finding; where a claim should generalize, checked
on the full sample.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
import config
plt.rcParams["figure.figsize"]=(11,5)

monthly = pd.read_parquet(config.DATA / "commodities_monthly.parquet")
CORE=["gold","silver","wti","brent","natgas","copper","palladium","platinum"]
ret = np.log(monthly[CORE].where(monthly[CORE]>0)).diff()
ret_ex = ret.loc[:config.EXPLORE_END]
print("core returns:", ret.shape, "| explore:", ret_ex.shape)

## Part 1a — Which assets drive the volatility regime?

Rebuild the regime, then decompose: in high-vol months, which commodities are
contributing most of the cross-sectional volatility? Energy-led vs metals-led
regimes imply different trades.

In [ ]:
vol6 = ret.rolling(6).std()                 # per-asset 6m vol
mkt_vol = ret.rolling(12).std().mean(axis=1) # market vol proxy
cutoff = mkt_vol.loc[:config.EXPLORE_END].quantile(0.70)
regime = (mkt_vol > cutoff).map({True:"high_vol", False:"calm"})

# per-asset average vol in each regime -> who drives high-vol?
drivers = vol6.groupby(regime).mean().T
drivers["ratio"] = drivers["high_vol"] / drivers["calm"]
drivers.sort_values("ratio", ascending=False).round(4)

**Read:** the `ratio` column ranks how much each asset's volatility inflates
in high-vol regimes. High ratio = this asset's turbulence defines the regime;
near-1 = it stays calm even when the market doesn't. Tells the trader *what kind*
of turbulence to expect.

## Part 1b — How persistent is a regime? (expected duration)

In [ ]:
# find contiguous runs of each regime, measure lengths
r = regime.dropna()
runs=[]; cur=r.iloc[0]; length=1
for v in r.iloc[1:]:
    if v==cur: length+=1
    else: runs.append((cur,length)); cur=v; length=1
runs.append((cur,length))
runs_df = pd.DataFrame(runs, columns=["regime","months"])
print("High-vol regime durations (months):")
print(runs_df[runs_df.regime=="high_vol"]["months"].describe().round(1).to_string())
print("\nCalm regime durations (months):")
print(runs_df[runs_df.regime=="calm"]["months"].describe().round(1).to_string())
print(f"\nOnce high-vol starts, median duration: "
      f"{runs_df[runs_df.regime=='high_vol']['months'].median():.0f} months")

**Dashboard value:** "once turbulence starts it typically lasts N months"
tells the trader how long to stay defensive. Far more useful than a bare flag.

## Part 1c — Regime transition probabilities

In [ ]:
# P(next regime | current regime) -- how sticky is each state?
trans = pd.crosstab(regime.shift(1), regime, normalize="index")
print("Transition matrix  P(next | current):")
print(trans.round(3).to_string())
print(f"\nP(stay high-vol | in high-vol): {trans.loc['high_vol','high_vol']:.2f}")
print(f"P(stay calm | in calm):         {trans.loc['calm','calm']:.2f}")

## Part 2 — Lead-lag structure

For each ordered pair (X, Y), correlate X's return THIS month with Y's return
NEXT month. A strong off-diagonal value means X tends to lead Y by a month.

**Leakage note:** this is descriptive structure on historical data, not a
tradable forecast — but we still build it cleanly (X at t vs Y at t+1) and read
it as "historically, X moves have preceded Y moves," with appropriate caution
about whether it persists.

In [ ]:
def lead_lag_matrix(returns, lag=1):
    cols = returns.columns
    M = pd.DataFrame(index=cols, columns=cols, dtype=float)  # rows=leader, cols=follower
    for x in cols:
        for y in cols:
            M.loc[x,y] = returns[x].corr(returns[y].shift(-lag))
    return M.astype(float)

ll = lead_lag_matrix(ret_ex, lag=1)
fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(ll, cmap="RdBu_r", vmin=-0.4, vmax=0.4)
ax.set_xticks(range(len(CORE))); ax.set_xticklabels(CORE,rotation=45,ha="right")
ax.set_yticks(range(len(CORE))); ax.set_yticklabels(CORE)
ax.set_xlabel("FOLLOWER (t+1)"); ax.set_ylabel("LEADER (t)")
for i in range(len(CORE)):
    for j in range(len(CORE)):
        ax.text(j,i,f"{ll.iloc[i,j]:.2f}",ha="center",va="center",fontsize=8)
plt.colorbar(im,fraction=0.046); plt.title("Lead-lag: leader(t) return vs follower(t+1) return")
plt.tight_layout(); plt.show()

**Read the OFF-DIAGONAL, asymmetrically.** If `ll[copper, X]` (copper leads
X) is notably larger than `ll[X, copper]` (X leads copper), copper leads that
relationship. The diagonal is each asset's own autocorrelation (usually weak for
monthly returns). Compare a cell to its transpose to find genuine lead-lag
asymmetry rather than plain co-movement.

In [ ]:
# extract the strongest ASYMMETRIC lead-lag pairs (X leads Y more than Y leads X)
pairs=[]
for i,x in enumerate(CORE):
    for j,y in enumerate(CORE):
        if x==y: continue
        asym = ll.loc[x,y] - ll.loc[y,x]   # positive => x leads y net
        pairs.append({"leader":x,"follower":y,"x_leads_y":round(ll.loc[x,y],3),
                      "y_leads_x":round(ll.loc[y,x],3),"asymmetry":round(asym,3)})
pairs_df = pd.DataFrame(pairs).sort_values("asymmetry",ascending=False)
print("Strongest net lead-lag relationships (leader tends to move a month before follower):")
pairs_df.head(10).to_string(index=False)

## Part 2b — Does the strongest lead-lag hold out of sample?

Any lead-lag we'd surface on the dashboard should survive 2022+, or it's an
explore-period artifact. Check the top pair.

In [ ]:
top = pairs_df.iloc[0]
lx, ly = top["leader"], top["follower"]
full_lead = ret[lx].corr(ret[ly].shift(-1))
ho_lead   = ret.loc[config.HOLDOUT_START:][lx].corr(ret.loc[config.HOLDOUT_START:][ly].shift(-1))
ex_lead   = ret_ex[lx].corr(ret_ex[ly].shift(-1))
print(f"Top pair: {lx} leads {ly}")
print(f"  explore corr(t, t+1): {ex_lead:+.3f}")
print(f"  holdout corr(t, t+1): {ho_lead:+.3f}")
print(f"  full-sample:          {full_lead:+.3f}")
print("\nIf holdout sign matches explore, the lead-lag is stable enough to show.")
print("If it flips or vanishes, flag it as explore-only and DON'T feature it.")

## Intelligence summary (fill after running)

**Regime character:**
- Driver assets (highest high-vol/calm ratio): ?
- Median high-vol duration: ? months
- Regime stickiness P(stay|state): ?

**Lead-lag:**
- Strongest net leader→follower: ?
- Holds out of sample? ?

**Dashboard implication:** the regime flag becomes "elevated vol likely, [energy/
metals]-driven, typically ~N months, currently [starting/persisting]." Lead-lag
adds "X moves have historically preceded Y" context — clearly labeled descriptive,
not a signal. Together: real market-structure intelligence, no false precision.